## Michigan DNR Trail Data Audit

This notebook validates the downloaded Upper Peninsula hiking-trail segments, explore placeholder values so they can be replaced in prep_data.py

In [1]:
from pathlib import Path
import sys

import pandas as pd
import geopandas as gpd

sys.path.append("..")

from src.trails import (
    prep_columns,
    replace_missing_placeholders,
    aggregate_column
)


DATA_PATH = Path("../data/raw/dnr_up_hiking_trails.geojson")
PROCESSED_PATH = Path("../data/processed/dnr_up_hiking_segments_clean.parquet")

trails = gpd.read_file(DATA_PATH)

print(f"Rows: {len(trails):,}")
print(f"Columns: {len(trails.columns)}")
print(f"CRS: {trails.crs}")
trails.head(10)

Rows: 2,437
Columns: 15
CRS: EPSG:4326


,OBJECTID,DNRTrail,TrailNamePrimary,HikingName,FacilityName,County,Peninsula,Hiking,TrailUseCategory,OpenClosedStatusNonmotor,SurfaceType,TrailWidthFeet,ADAAccessible,SegmentLengthMiles,geometry
0,1919,DNR Trail,Straits - Main Trail,Straits Main Trail,Straits State Park,Mackinac,Upper Peninsula,Hiking,Nonmotorized,Open,Dirt Natural,0 To 2 Feet,Not Accessible,0.006157,"LINESTRING (-84.72138 45.84982, -84.72135 45.8..."
1,2486,DNR Trail,Little Presque Isle Multi Use Pathway,Little Presque Isle Multi Use Pathway,State Forest,Marquette,Upper Peninsula,Hiking,Nonmotorized,Temporarily Closed,Dirt Natural,-1,Not Accessible,1.200800,"LINESTRING (-87.47649 46.63472, -87.47644 46.6..."
2,2495,DNR Trail,UP 2,North Country Trail,-1,Mackinac,Upper Peninsula,Hiking,Multi Use Motorized Nonmotorized,Open,Dirt Natural,12 Feet And Over,Not Accessible,3.891672,"LINESTRING (-84.73683 45.87386, -84.73687 45.8..."
3,2540,DNR Trail,Gemini Lake Pathway,Gemini Lake Pathway,State Forest,Schoolcraft,Upper Peninsula,Hiking,Nonmotorized,Open,Dirt Natural,0 To 2 Feet,Not Accessible,0.514227,"LINESTRING (-86.30655 46.48498, -86.30623 46.4..."
4,2541,DNR Trail,Gemini Lake Pathway,Gemini Lake Pathway,State Forest,Schoolcraft,Upper Peninsula,Hiking,Nonmotorized,Open,Dirt Natural,0 To 2 Feet,Not Accessible,0.264824,"LINESTRING (-86.30464 46.48159, -86.30522 46.4..."
5,2565,DNR Trail,North Country Trail,North Country Trail,-1,Alger,Upper Peninsula,Hiking,Nonmotorized,Open,Dirt Natural,0 To 2 Feet,Not Accessible,0.126151,"LINESTRING (-85.93435 46.66256, -85.93413 46.6..."
6,2566,DNR Trail,North Country Trail,North Country Trail,-1,Marquette,Upper Peninsula,Hiking,Nonmotorized,Open,Dirt Natural,0 To 2 Feet,Not Accessible,0.631410,"LINESTRING (-87.54912 46.66508, -87.54912 46.6..."
7,2567,DNR Trail,North Country Trail,North Country Trail,-1,Marquette,Upper Peninsula,Hiking,Nonmotorized,Open,Dirt Natural,0 To 2 Feet,Not Accessible,0.392742,"LINESTRING (-87.57158 46.66467, -87.5716 46.66..."
8,2580,DNR Trail,North Country Trail,Porcupine Mountain Wilderness Lake Superior Trail,Porcupine Mountains Wilderness State Park,Gogebic,Upper Peninsula,Hiking,Nonmotorized,Open,Dirt Natural,0 To 2 Feet,Not Accessible,0.034613,"LINESTRING (-89.93643 46.72285, -89.93636 46.7..."
9,2581,DNR Trail,Porcupine Mts Lily Pond Trail,Porcupine Mts Lily Pond Trail,Porcupine Mountains Wilderness State Park,Ontonagon,Upper Peninsula,Hiking,Nonmotorized,Open,Dirt Natural,0 To 2 Feet,Not Accessible,0.974193,"LINESTRING (-89.76689 46.73512, -89.76639 46.7..."


In [2]:
trails.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 2437 entries, 0 to 2436
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   OBJECTID                  2437 non-null   int32   
 1   DNRTrail                  2437 non-null   str     
 2   TrailNamePrimary          2437 non-null   str     
 3   HikingName                2437 non-null   str     
 4   FacilityName              2437 non-null   str     
 5   County                    2437 non-null   str     
 6   Peninsula                 2437 non-null   str     
 7   Hiking                    2437 non-null   str     
 8   TrailUseCategory          2437 non-null   str     
 9   OpenClosedStatusNonmotor  2437 non-null   str     
 10  SurfaceType               2437 non-null   str     
 11  TrailWidthFeet            2437 non-null   str     
 12  ADAAccessible             2437 non-null   str     
 13  SegmentLengthMiles        2437 non-null 

In [3]:
mask = (trails["OpenClosedStatusNonmotor"] == "Temporarily Closed")

trails.loc[mask]

,OBJECTID,DNRTrail,TrailNamePrimary,HikingName,FacilityName,County,Peninsula,Hiking,TrailUseCategory,OpenClosedStatusNonmotor,SurfaceType,TrailWidthFeet,ADAAccessible,SegmentLengthMiles,geometry
1,2486,DNR Trail,Little Presque Isle Multi Use Pathway,Little Presque Isle Multi Use Pathway,State Forest,Marquette,Upper Peninsula,Hiking,Nonmotorized,Temporarily Closed,Dirt Natural,-1,Not Accessible,1.200800,"LINESTRING (-87.47649 46.63472, -87.47644 46.6..."
112,3514,DNR Trail,Little Presque Isle Multi Use Pathway,Little Presque Isle Multi Use Pathway,State Forest,Marquette,Upper Peninsula,Hiking,Nonmotorized,Temporarily Closed,Dirt Natural,-1,Not Accessible,0.445860,"LINESTRING (-87.49289 46.63979, -87.49189 46.6..."
113,3515,DNR Trail,Little Presque Isle Multi Use Pathway,Little Presque Isle Multi Use Pathway,State Forest,Marquette,Upper Peninsula,Hiking,Nonmotorized,Temporarily Closed,Dirt Natural,-1,Not Accessible,0.259620,"LINESTRING (-87.48695 46.64034, -87.48692 46.6..."
114,3516,DNR Trail,Little Presque Isle Multi Use Pathway,Little Presque Isle Multi Use Pathway,State Forest,Marquette,Upper Peninsula,Hiking,Nonmotorized,Temporarily Closed,Dirt Natural,-1,Not Accessible,0.432088,"LINESTRING (-87.48435 46.63729, -87.48407 46.6..."
117,3521,DNR Trail,Little Presque Isle Multi Use Pathway,Little Presque Isle Multi Use Pathway,State Forest,Marquette,Upper Peninsula,Hiking,Nonmotorized,Temporarily Closed,Dirt Natural,-1,Not Accessible,0.224164,"LINESTRING (-87.46916 46.63441, -87.46845 46.6..."
119,3524,DNR Trail,Little Presque Isle Multi Use Pathway,Little Presque Isle Multi Use Pathway,State Forest,Marquette,Upper Peninsula,Hiking,Nonmotorized,Temporarily Closed,Dirt Natural,-1,Not Accessible,1.786506,"LINESTRING (-87.47457 46.63158, -87.47459 46.6..."
2025,12306,DNR Trail,Little Presque Isle Multi Use Pathway,Little Presque Isle Multi Use Pathway,State Forest,Marquette,Upper Peninsula,Hiking,Nonmotorized,Temporarily Closed,Dirt Natural,-1,Not Accessible,0.023261,"LINESTRING (-87.47832 46.6062, -87.47822 46.60..."


In [4]:
trails.isna().sum().sort_values(ascending=False)

OBJECTID                    0
DNRTrail                    0
TrailNamePrimary            0
HikingName                  0
FacilityName                0
County                      0
Peninsula                   0
Hiking                      0
TrailUseCategory            0
OpenClosedStatusNonmotor    0
SurfaceType                 0
TrailWidthFeet              0
ADAAccessible               0
SegmentLengthMiles          0
geometry                    0
dtype: int64

In [5]:
unique_counts = (
    trails.nunique(dropna=False)
    .rename("unique_count")
    .reset_index()
    .rename(columns = {"index":"column"})
)

unique_counts

,column,unique_count
0,OBJECTID,2437
1,DNRTrail,1
2,TrailNamePrimary,119
3,HikingName,135
4,FacilityName,25
5,County,15
6,Peninsula,1
7,Hiking,1
8,TrailUseCategory,2
9,OpenClosedStatusNonmotor,4


In [6]:
PLACEHOLDER_VALUES = {
    "",
    "-1",
    "-2",
    "99",
    "-99",
    "Unspecified",
    "Unknown",
    "None",
    "N/A",
    "<NA>",
    "NA"
}

text_columns = trails.select_dtypes(
    include=["object","string"]
).columns

placeholder_records = []

for column in text_columns:
    normalized = trails[column].astype("string").str.strip()
    matches = normalized.isin(PLACEHOLDER_VALUES)

    placeholder_records.append(
        {
            "column": column,
            "placeholder_count": int(matches.sum()),
            "placeholder_percent": round(matches.mean() * 100, 2),
            "values_found": sorted(
                normalized.loc[matches].dropna().unique().tolist()
            ),
        }
    )

placeholder_audit = (
    pd.DataFrame(placeholder_records)
    .sort_values("placeholder_count", ascending=False)
    .reset_index(drop=True)
)

placeholder_audit

,column,placeholder_count,placeholder_percent,values_found
0,FacilityName,658,27.00,"[-1, Unspecified]"
1,TrailWidthFeet,36,1.48,[-1]
2,ADAAccessible,33,1.35,[-99]
3,SurfaceType,13,0.53,[-1]
4,HikingName,0,0.00,[]
5,TrailNamePrimary,0,0.00,[]
6,DNRTrail,0,0.00,[]
7,County,0,0.00,[]
8,TrailUseCategory,0,0.00,[]
9,Hiking,0,0.00,[]


In [7]:
cleaned_data = replace_missing_placeholders(trails)


hiking_name_audit = (
    cleaned_data.groupby("HikingName", dropna=False)
    .agg(
        segment_count=("OBJECTID", "size"),
        county_count=("County", "nunique"),
        facility_count=("FacilityName","nunique"),
        primary_name_count=("TrailNamePrimary","nunique"),
        counties=(
            "County",
            lambda values: sorted(
                values.dropna().astype(str).unique()
            ),
        ),
        facilities=(
            "FacilityName",
            lambda values: sorted(
                values.dropna().astype(str).unique()   
            ),
        ),
    )
    .sort_values(
        ["county_count", "facility_count", "segment_count"],
        ascending=False,
    )
)

print(f"Unique hiking names: {len(hiking_name_audit)}:,")
hiking_name_audit.head(30)

Unique hiking names: 135:,


,segment_count,county_count,facility_count,primary_name_count,counties,facilities
HikingName,,,,,,
North Country Trail,597,9,6,10,['Alger' 'Baraga' 'Chippewa' 'Gogebic' 'Hought...,['Craig Lake State Park' 'Porcupine Mountains ...
Iron Belle Trail,15,7,1,4,['Delta' 'Luce' 'Mackinac' 'Marquette' 'Menomi...,['Porcupine Mountains Wilderness State Park']
Porcupine Mountain Wilderness Lake Superior Trail,128,2,1,3,['Gogebic' 'Ontonagon'],['Porcupine Mountains Wilderness State Park']
Porcupine Mts Big Carp River Trail,78,2,1,2,['Gogebic' 'Ontonagon'],['Porcupine Mountains Wilderness State Park']
Porcupine Mountain Wilderness - Lake Superior Trail,72,2,1,1,['Gogebic' 'Ontonagon'],['Porcupine Mountains Wilderness State Park']
Fox River Pathway,24,2,1,2,['Alger' 'Schoolcraft'],['State Forest']
Tahquamenon Falls State Park Lower Falls Foot Trails,22,2,1,2,['Chippewa' 'Luce'],['Tahquamenon Falls State Park']
Tahquamenon Falls River Trail,14,2,1,1,['Chippewa' 'Luce'],['Tahquamenon Falls State Park']
Porcupine Mountain Wilderness Cross Trail Correction Line Trail,12,2,1,2,['Gogebic' 'Ontonagon'],['Porcupine Mountains Wilderness State Park']


In [8]:
prep_data =prep_columns(cleaned_data)

assert len(prep_data) == len(trails)
assert prep_data["OBJECTID"].is_unique
assert prep_data["TrailGroupName"].notna().all()

In [9]:
width_counts = (
    prep_data
    .groupby("TrailGroupName")["TrailWidthFeet"]
    .nunique(dropna=True)
)

width_counts

TrailGroupName
Alger | Fox River Pathway                       1
Alger | Laughing Whitefish Falls - Trails       1
Alger | Laughing Whitefish Falls Trails         0
Alger | North Country Trail                     2
Alger | Tyoga Historical Pathway                1
                                               ..
Schoolcraft | Indian Lake Dufour Creek Trail    1
Schoolcraft | Indian Lake Pathway               1
Schoolcraft | Indian Lake Trails                2
Schoolcraft | Iron Belle Trail                  1
Schoolcraft | Steeb Pathway                     1
Name: TrailWidthFeet, Length: 159, dtype: int64

In [10]:
trail_group_summary = (
    prep_data.groupby("TrailGroupName")
    .agg(
        segment_count=("OBJECTID", "size"),
        reported_length_miles=("SegmentLengthMiles", "sum"),
        facility_count=("FacilityName", "nunique"),
        surface_type=("SurfaceType", aggregate_column),
        status_count=("OpenClosedStatusNonmotor", "nunique"),
        trail_width=("TrailWidthFeet", aggregate_column),
    )
    .sort_values("reported_length_miles", ascending=False)
)

trail_group_summary.head(30)

,segment_count,reported_length_miles,facility_count,surface_type,status_count,trail_width
TrailGroupName,,,,,,
Alger | North Country Trail,129,96.990313,1,Varies,1,Varies
Marquette | North Country Trail,60,74.256774,1,Varies,1,Varies
Ontonagon | North Country Trail,100,73.306963,1,Varies,1,Varies
Chippewa | North Country Trail,148,66.307730,1,Varies,1,Varies
Mackinac | Iron Belle Trail,1,63.804200,0,Asphalt,1,0 To 2 Feet
Delta | Iron Belle Trail,6,52.355545,0,Asphalt,1,0 To 2 Feet
Marquette | Iron Ore Heritage Trail,49,50.502948,0,Varies,1,Varies
Baraga | North Country Trail,21,47.588697,1,Varies,1,Varies
Gogebic | North Country Trail,24,43.997004,1,Varies,1,Varies
